# Evaluate a saved BioASQ retrieval subset

This notebook loads one subset created by `BioASQ_sample.ipynb` and evaluates a sentence-transformer on it. Change `SAMPLE_NAME` to compare the six question-type and document-count subsets.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("/content/retrieval-benchlab")
if "google.colab" in sys.modules:
    if not (REPO_ROOT / ".git").exists():
        !git clone --depth 1 https://github.com/lohex/retrieval-benchlab.git {REPO_ROOT}
    %cd {REPO_ROOT}
else:
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))

!pip -q install -U datasets sentence-transformers


In [ ]:
import logging

import torch
from sentence_transformers import SentenceTransformer, util
from sentence_transformers.sentence_transformer.evaluation import InformationRetrievalEvaluator

from src.io import load_bioasq_sample, mount_google_drive, sample_directory

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)
logger = logging.getLogger("retrieval-test")


## Configuration

Available subset names are `list-one`, `list-multiple`, `factoid-one`, `factoid-multiple`, `summary-one`, and `summary-multiple`.

In [ ]:
SAMPLE_ROOT = "/content/drive/MyDrive/Retreaval/data"
SAMPLE_NAME = "list-multiple"
SAMPLE_DIR = sample_directory(SAMPLE_ROOT, SAMPLE_NAME)

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
BATCH_SIZE = 64
CORPUS_CHUNK_SIZE = 10_000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
logger.info("Device: %s", DEVICE)


## Load and validate the saved subset

In [ ]:
mount_google_drive()
queries, relevant_docs, corpus, sample_metadata = load_bioasq_sample(SAMPLE_DIR)

print(f"Subset: {sample_metadata['subset_name']}")
print(f"Queries: {len(queries):,}")
print(f"Relevant relations: {sum(len(ids) for ids in relevant_docs.values()):,}")
print(f"Corpus documents: {len(corpus):,}")


## Encode and evaluate

Cosine similarity is set explicitly as the retrieval score. Ranking metrics compare the resulting ranks with every gold document associated with each query.

In [ ]:
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
logger.info("Model: %s", MODEL_NAME)


In [ ]:
evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    corpus_chunk_size=CORPUS_CHUNK_SIZE,
    mrr_at_k=[10],
    ndcg_at_k=[10],
    accuracy_at_k=[1, 3, 5, 10, 100],
    precision_recall_at_k=[1, 3, 5, 10, 100],
    map_at_k=[100],
    batch_size=BATCH_SIZE,
    name=f"bioasq-11b-{SAMPLE_NAME}",
    show_progress_bar=True,
    write_csv=False,
    score_functions={"cosine": util.cos_sim},
    main_score_function="cosine",
)

results = evaluator(model)
requested_names = ("ndcg@10", "mrr@10", "recall@10", "recall@100", "map@100")
requested_metrics = {
    key: value
    for key, value in results.items()
    if any(name in key.lower() for name in requested_names)
}

print("Requested metrics:")
for key, value in sorted(requested_metrics.items()):
    print(f"{key}: {value:.4f}")

if len(requested_metrics) < len(requested_names):
    print("\nAll returned metrics for API diagnostics:")
    for key, value in sorted(results.items()):
        print(f"{key}: {value}")
